In [1]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
from urllib.parse import urlparse
import datetime


Open CSV file obtained by PRAW

In [2]:
df = pd.read_csv(r'C:\Users\Khoi Vu\Desktop\Buildapcsales_rawdata.csv')

df.head(5)

,Post ID,Title,Url,Author,Score,Publish Date,Total No. of Comments,Permalink,Flair
0,ko6d4q,"[Monitor] Acer Predator XB273U GXbmiipruzx 27""...",https://store.acer.com/en-us/monitors/gaming/2...,floppelganger,1,2021-01-01 00:10:40,80,/r/buildapcsales/comments/ko6d4q/monitor_acer_...,Monitor
1,ko6mzk,"[Laptop]Open Box Excellent Dell - G5 15.6"" Gam...",https://www.bestbuy.com/site/dell-g5-15-6-gami...,crownpuff,1,2021-01-01 00:30:30,67,/r/buildapcsales/comments/ko6mzk/laptopopen_bo...,Laptop
2,ko7flm,"[AIO] CORSAIR iCUE H115i RGB PRO XT - $98,99 (...",https://www.newegg.com/corsair-icue-h115i-rgb-...,TheZeR0x,1,2021-01-01 01:37:58,7,/r/buildapcsales/comments/ko7flm/aio_corsair_i...,Cooler
3,ko7ln0,[Motherboard] ASUS ROG Crosshair VIII Dark Her...,https://www.newegg.com/asus-rog-crosshair-viii...,imaginary_num6er,1,2021-01-01 01:53:58,42,/r/buildapcsales/comments/ko7ln0/motherboard_a...,Motherboard
4,ko7qh5,"[Monitor] ASUS TUF GAMING VG27WQ 27"" WQHD 2560...",https://www.newegg.com/black-asus-tuf-gaming-v...,crownpuff,1,2021-01-01 02:06:23,58,/r/buildapcsales/comments/ko7qh5/monitor_asus_...,Monitor


Dataframe clean up 

In [3]:
#Split up Title column to get item type
df[['Type','Title']] = df.Title.str.split(expand=True,n=1, pat=']') 
df[['1','Type']] = df.Type.str.split(expand=True,n=1, pat='[')

#Use urlparse to get website of URL column
df["Site"] =df["Url"].astype(str).apply(lambda x: urlparse(x).netloc)

#Change Type to all uppercase
df['Type'] = df['Type'].str.upper()
df['Title'] = df['Title'].str.upper()

#Change Publish Date to date format
df = df.rename(columns={'Publish Date' :'Date'})
df['Date'] = pd.to_datetime(df.Date).dt.normalize()

#Delete reposts in same day by Date+URL
df = df.drop_duplicates(subset = ['Date', 'Url'],keep = 'first').reset_index(drop = True)
df = df.drop_duplicates(subset = ['Date', 'Title'],keep = 'first').reset_index(drop = True)

#Use these columns
df = df[['Date','Type','Title','Site','Url','Author','Score','Total No. of Comments']]


df.head(5)

,Date,Type,Title,Site,Url,Author,Score,Total No. of Comments
0,2021-01-01,MONITOR,"ACER PREDATOR XB273U GXBMIIPRUZX 27"" WQHD 270...",store.acer.com,https://store.acer.com/en-us/monitors/gaming/2...,floppelganger,1,80
1,2021-01-01,LAPTOP,"OPEN BOX EXCELLENT DELL - G5 15.6"" GAMING LAPT...",www.bestbuy.com,https://www.bestbuy.com/site/dell-g5-15-6-gami...,crownpuff,1,67
2,2021-01-01,AIO,"CORSAIR ICUE H115I RGB PRO XT - $98,99 ($104....",www.newegg.com,https://www.newegg.com/corsair-icue-h115i-rgb-...,TheZeR0x,1,7
3,2021-01-01,MOTHERBOARD,ASUS ROG CROSSHAIR VIII DARK HERO AM4 AMD X57...,www.newegg.com,https://www.newegg.com/asus-rog-crosshair-viii...,imaginary_num6er,1,42
4,2021-01-01,MONITOR,"ASUS TUF GAMING VG27WQ 27"" WQHD 2560 X 1440 1...",www.newegg.com,https://www.newegg.com/black-asus-tuf-gaming-v...,crownpuff,1,58


Begin EDA

Top 20 Types

In [4]:
top20_type = df['Type'].value_counts().head(20)
fig_toptype = px.bar(top20_type, title='Top 20 Sites by # of posts in 2021')
fig_toptype.show()

Top 20 Sites

In [5]:
top20_site = df['Site'].value_counts().head(20)
fig_topsite = px.bar(top20_site, title='Top 20 Sites by # of posts in 2021')
fig_topsite.show()

Top 20 Authors

In [6]:
top20_authors = df['Author'].value_counts().head(20)
px.bar(top20_authors,title='Top 20 Authors by # of posts 2021')

Posts by Date

In [7]:
px.histogram(df,'Date', title='# of posts by Date')

In [8]:
pio.renderers.default = "svg"